In [ ]:
import sys
sys.path.insert(0, '..')
from src.search.engine import LegalSemanticSearchEngine
from src.loader import load_json

In [ ]:
# Загружаем статьи
raw_D = load_json("/home/gshjis/Python_projects/IA_NPA_auditor/data/processed/laws.json")
articles = [r["content"] for r in raw_D]

print(f"Загружено статей: {len(articles)}")


In [3]:
tagger = LegalSemanticSearchEngine(
    tags_filepath="/home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json", 
    training_corpus=articles,
    tags_per_article=100
)

2026-03-16 14:33:11,740 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Вычисление схожести с тегами...
2026-03-16 14:33:11,804 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загрузка эмбеддингов из data/cache/article_embeddings.npy...
2026-03-16 14:33:11,943 - IA_NPA_auditor - INFO - ℹ️ ℹ️ IDF веса вычислены
2026-03-16 14:33:11,944 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Инициализация завершена


In [7]:
diverse_queries = [
    "Вырубка леса запрещена",
    # # ===== КОНСТИТУЦИЯ =====
    # "Какие личные права и свободы гарантированы гражданам?",
    # "В каких случаях допускается ограничение прав и свобод?",
    # "Какие существуют политические права граждан?",
    # "Что такое право на неприкосновенность частной жизни?",
    # "Какие гарантии равенства перед законом предусмотрены?",
    # "Кто может быть избран президентом и каковы его полномочия?",
    # "Как формируется парламент и каковы его функции?",
    # "Что такое правительство и чем оно занимается?",
    # "Какова роль Конституционного Суда?",
    # "Что такое местное самоуправление?",
    # "Кто имеет право избирать и быть избранным?",
    # "Что такое референдум?",
    # "Кто является гражданином?",
    # "Можно ли лишить гражданства?",
    # "Какие конституционные обязанности есть у граждан?",
    # "Что такое воинская обязанность?",
    # "Каковы обязанности по охране природы?",
    # "Что такое чрезвычайное положение?",
    
    # # ===== ТРУДОВОЙ КОДЕКС =====
    # "Какие трудовые права гарантированы работникам?",
    # "Что такое право на справедливое вознаграждение за труд?",
    # "Какая продолжительность рабочего времени установлена законом?",
    # "Что говорит закон о праве на отдых и отпуск?",
    # "Как регулируется труд женщин и несовершеннолетних?",
    # "Что такое трудовой договор и как он заключается?",
    # "Каковы основания для увольнения работника?",
    # "Что такое дисциплинарная ответственность?",
    # "Как регулируется сверхурочная работа?",
    # "Какие гарантии при сокращении штата?",
    # "Что такое вынужденный прогул?",
    # "Как оплачивается больничный?",
    # "Каков порядок расторжения трудового договора?",
    # "Что такое испытательный срок?",
    # "Какие гарантии для беременных женщин?",
]


In [8]:
for idx, query in enumerate(diverse_queries, 1):
    print("\n" + "=" * 80)
    print(f"📌 ЗАПРОС №{idx}".center(80))
    print("=" * 80)
    print(f"{query}\n")
    
    results = tagger.find_articles_by_new_sentence(query, k=10)
    
    for rank, res in enumerate(results, 1):
        # Определяем иконку по score
        if res['score'] >= 0.6:
            icon = "🟢🔝"
        elif res['score'] >= 0.5:
            icon = "🟡"
        else:
            icon = "⚪"
        
        # Основная информация
        print(f"{icon} [{rank}] Score: {res['score']:.3f}")
        
        # Текст статьи (первые 150 символов)
        article_text = res['article']['content'] if isinstance(res['article'], dict) else str(res['article'])
        print(f"   📄 {article_text[:150]}...")
        
        # ТЕГИ: топ-5 тегов запроса
        print(f"   🔍 Теги запроса (топ-5):", end=" ")
        query_tags_top = sorted(res['query_tags'].items(), key=lambda x: x[1], reverse=True)[:5]
        for tag, weight in query_tags_top:
            print(f"{tag}({weight:.2f})", end=" ")
        print()
        
        # ТЕГИ: топ-5 общих тегов
        print(f"   🤝 Общие теги ({len(res['common_tags'])}):", end=" ")
        for tag in res['common_tags'][:5]:
            # берем вес из article_tags для этих тегов
            if tag in res['article_tags']:
                print(f"{tag}({res['article_tags'][tag]:.2f})", end=" ")
        print()
        
        # ТЕГИ: топ-3 уникальных тега статьи (не в запросе)
        article_unique = {tag: weight for tag, weight in res['article_tags'].items() 
                         if tag not in res['query_tags']}
        if article_unique:
            print(f"   🏷️ Уникальные теги статьи (топ-3):", end=" ")
            for tag, weight in sorted(article_unique.items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"{tag}({weight:.2f})", end=" ")
            print()
        
        print()
    
    print("-" * 80)


                                  📌 ЗАПРОС №1                                   
Вырубка леса запрещена

2026-03-16 14:33:16,131 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Поиск статей для запроса: 'Вырубка леса запрещена...'
2026-03-16 14:33:16,133 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Получение рекомендаций тегов для текста длиной 22...
2026-03-16 14:33:16,296 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 100 тегов
2026-03-16 14:33:16,380 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 10 статей
🟢🔝 [1] Score: 0.676
   📄 Каждый имеет право на благоприятную окружающую среду и на возмещение вреда, причиненного нарушением этого права. Государство осуществляет контроль за ...
   🔍 Теги запроса (топ-5): цензура(0.52) ограничение_прав(0.50) сокращенное_рабочее_время(0.49) порядок_изменения_конституции(0.49) забастовка(0.48) 
   🤝 Общие теги (62): цензура(0.59) ограничение_прав(0.56) сокращенное_рабочее_время(0.49) забастовка(0.52) дисциплинарное_взыскание(0.51) 
   🏷️ Уникальные теги статьи (топ-3): патриотизм(0.60) эко

In [10]:
a = tagger.get_tag_recommendations_formatted("Вырубка леса запрещена")
print(a)

2026-03-16 14:33:26,897 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Получение рекомендаций тегов для текста длиной 22...
2026-03-16 14:33:27,046 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Найдено 100 тегов

🔍 ТЕГИ ДЛЯ ЗАПРОСА: 'Вырубка леса запрещена'
----------------------------------------
   цензура                        [██████████░░░░░░░░░░] 0.519
   ограничение_прав               [█████████░░░░░░░░░░░] 0.496
   сокращенное_рабочее_время      [█████████░░░░░░░░░░░] 0.491
   порядок_изменения_конституции  [█████████░░░░░░░░░░░] 0.490
   забастовка                     [█████████░░░░░░░░░░░] 0.484
   сокращение_численности_или_штата [█████████░░░░░░░░░░░] 0.477
   дисциплинарное_взыскание       [█████████░░░░░░░░░░░] 0.476
   судебная_защита                [█████████░░░░░░░░░░░] 0.467
   отстранение_от_работы          [█████████░░░░░░░░░░░] 0.465
   территориальная_целостность    [█████████░░░░░░░░░░░] 0.463
   нормальная_продолжительность   [█████████░░░░░░░░░░░] 0.460
   верховенство_права             [